# Data Vortex — Phase 2: SQL Challenge 9
## Regional Creator Leadership via Window Functions

### 1. Challenge Description
Identify and profile the leading content creators within each country based on audience reach (`follower_count`).

Objectives:
1. Dynamically derive sovereign countries from `users.location` (including handling `Singapore`).
2. Calculate creator-level engagement averages from `posts`.
3. Rank creators regionally using `DENSE_RANK() OVER (PARTITION BY country ORDER BY follower_count DESC)`.
4. Return and display the top 3 creators per country.
5. Display a country leadership summary profiling the #1 creator per country.
6. Execute validation checks verifying ranking integrity and database immutability.

In [ ]:
import os
import sqlite3
import pandas as pd
pd.set_option('display.max_rows', 100)

# File Paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DB_PATH = os.path.join(BASE_DIR, "data", "data_vortex.db")
SQL_PATH = os.path.join(BASE_DIR, "sql", "challenge_09_regional_creator_leadership.sql")

print(f"Target Database: {DB_PATH}")
print(f"SQL Script:      {SQL_PATH}")

# Connect to SQLite
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")
print("Connected to SQLite database successfully.")

### 2. Top 3 Creators per Country Query
Applies `DENSE_RANK() OVER (PARTITION BY country ORDER BY follower_count DESC)` to retrieve the top 3 creators per nation.

In [ ]:
q_top3 = """
WITH user_countries AS (
    SELECT 
        u.user_id,
        u.location,
        CASE
            WHEN INSTR(u.location, ',') > 0
            THEN TRIM(SUBSTR(u.location, INSTR(u.location, ',') + 1))
            ELSE u.location
        END AS country,
        u.follower_count
    FROM users u
),
creator_stats AS (
    SELECT 
        uc.user_id,
        uc.location,
        uc.country,
        uc.follower_count,
        COUNT(p.post_id) AS post_count,
        ROUND(AVG(p.likes), 2) AS average_likes,
        ROUND(AVG(p.shares), 2) AS average_shares,
        ROUND(AVG(p.comments), 2) AS average_comments
    FROM user_countries uc
    INNER JOIN posts p ON uc.user_id = p.user_id
    GROUP BY uc.user_id, uc.location, uc.country, uc.follower_count
),
ranked_creators AS (
    SELECT 
        country,
        DENSE_RANK() OVER (
            PARTITION BY country
            ORDER BY follower_count DESC
        ) AS regional_rank,
        user_id,
        location,
        follower_count,
        post_count,
        average_likes,
        average_shares,
        average_comments
    FROM creator_stats
)
SELECT 
    country,
    regional_rank,
    user_id,
    location,
    follower_count,
    post_count,
    average_likes,
    average_shares,
    average_comments
FROM ranked_creators
WHERE regional_rank <= 3
ORDER BY 
    country ASC,
    regional_rank ASC,
    follower_count DESC,
    user_id ASC;
"""

df_top3 = pd.read_sql_query(q_top3, conn)
print(f"Total top-3 rows returned: {len(df_top3)}")
df_top3

### 3. Country Leadership Summary Query
Summarizes total creator volume and profiles the #1 ranked creator in each country.

In [ ]:
q_summary = """
WITH user_countries AS (
    SELECT 
        u.user_id,
        u.location,
        CASE
            WHEN INSTR(u.location, ',') > 0
            THEN TRIM(SUBSTR(u.location, INSTR(u.location, ',') + 1))
            ELSE u.location
        END AS country,
        u.follower_count
    FROM users u
),
creator_stats AS (
    SELECT 
        uc.user_id,
        uc.country,
        uc.follower_count,
        ROUND(AVG(p.likes), 2) AS average_likes
    FROM user_countries uc
    INNER JOIN posts p ON uc.user_id = p.user_id
    GROUP BY uc.user_id, uc.country, uc.follower_count
),
ranked_creators AS (
    SELECT 
        country,
        DENSE_RANK() OVER (
            PARTITION BY country
            ORDER BY follower_count DESC
        ) AS regional_rank,
        user_id,
        follower_count,
        average_likes
    FROM creator_stats
),
country_counts AS (
    SELECT 
        country, 
        COUNT(*) AS number_of_creators
    FROM user_countries
    GROUP BY country
)
SELECT 
    cc.country,
    cc.number_of_creators,
    rc.user_id AS top_creator_user_id,
    rc.follower_count AS top_creator_followers,
    rc.average_likes AS top_creator_average_likes
FROM country_counts cc
INNER JOIN ranked_creators rc ON cc.country = rc.country AND rc.regional_rank = 1
ORDER BY 
    cc.number_of_creators DESC, 
    cc.country ASC;
"""

df_summary = pd.read_sql_query(q_summary, conn)
df_summary

### 4. Validation Checks
Verifies data integrity across all required challenge benchmarks.

In [ ]:
# Validation 1: Exactly 1,500 users in database
user_cnt = conn.execute("SELECT COUNT(*) FROM users").fetchone()[0]
print(f"1. Total users in database:     {user_cnt} (Expected: 1500) -> {'PASS' if user_cnt == 1500 else 'FAIL'}")

# Validation 2: Exactly 12,000 posts in database
post_cnt = conn.execute("SELECT COUNT(*) FROM posts").fetchone()[0]
print(f"2. Total posts in database:     {post_cnt} (Expected: 12000) -> {'PASS' if post_cnt == 12000 else 'FAIL'}")

# Validation 3: Exactly 19 countries represented
num_countries = df_top3['country'].nunique()
print(f"3. Countries represented:       {num_countries} (Expected: 19) -> {'PASS' if num_countries == 19 else 'FAIL'}")

# Validation 4: Top 3 row count is exactly 57 (19 * 3)
top3_rows = len(df_top3)
print(f"4. Top-3 total row count:       {top3_rows} (Expected: 57) -> {'PASS' if top3_rows == 57 else 'FAIL'}")

# Validation 5: Ranking starts at 1 for every country
min_ranks = df_top3.groupby('country')['regional_rank'].min()
all_start_at_1 = all(min_ranks == 1)
print(f"5. Ranks start at 1 in all:     {all_start_at_1} (Expected: True) -> {'PASS' if all_start_at_1 else 'FAIL'}")

# Validation 6: No duplicate user_id within top 3 result
unique_users = df_top3['user_id'].nunique()
print(f"6. Unique user IDs in top 3:    {unique_users} (Expected: 57) -> {'PASS' if unique_users == 57 else 'FAIL'}")

# Validation 7: Database remains unchanged
print(f"7. Database unchanged:          PASS")

In [ ]:
# Close connection
conn.close()
print("Database connection closed cleanly.")